# Groundhog — GPU sweep (PyTorch on CUDA)
A **real** 10-run hyperparameter sweep: a small CNN on FashionMNIST, trained on the GPU, with a wide hyperparameter space so Groundhog derives clear insights (which knob actually moves accuracy) and tracks realistic **GPU-hours** from wall-clock. Every run records config, metrics, dataset, artifacts (checkpoint + logs), and cost.

In [1]:
import os, sys, json, time, urllib.request
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd())=='demo' else os.getcwd()
if os.path.basename(os.getcwd()) != 'demo':
    # allow running from repo root too
    ROOT = os.getcwd()
sys.path.insert(0, os.path.join(ROOT, 'sdk')); sys.path.insert(0, ROOT)
import groundhog
from demo.train_torch import train_and_evaluate, DATASET_INFO
import torch
print('CUDA available:', torch.cuda.is_available(), '|', (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'))

BACKEND = os.getenv('GROUNDHOG_API_URL', 'http://localhost:8000')
SIG = ['lr','batch_size','optimizer','weight_decay','dropout','conv_channels','hidden_dim','activation','epochs']
CACHE = os.path.join(ROOT, 'demo', '.gpu_project.json')

def _http(method, path, payload=None):
    data = json.dumps(payload).encode() if payload is not None else None
    req = urllib.request.Request(BACKEND.rstrip('/')+path, data=data, method=method, headers={'Content-Type':'application/json'})
    with urllib.request.urlopen(req, timeout=30) as r: return json.loads(r.read().decode())

def ensure_project():
    if os.path.exists(CACHE):
        pid = json.load(open(CACHE))['project_id']
        try: _http('GET', f'/api/projects/{pid}'); return pid
        except Exception: pass
    pid = _http('POST','/api/projects',{'name':'FashionMNIST CNN Sweep','significant_keys':SIG})['project_id']
    json.dump({'project_id':pid}, open(CACHE,'w'))
    print('created project', pid); return pid

project_id = ensure_project()
# optional: attach W&B creds from env so runs mirror out to W&B
if os.getenv('WANDB_PROJECT') and os.getenv('WANDB_API_KEY'):
    _http('POST', f'/api/projects/{project_id}/wandb', {'entity':os.getenv('WANDB_ENTITY'),'project':os.getenv('WANDB_PROJECT'),'api_key':os.getenv('WANDB_API_KEY')})
    print('W&B attached')
groundhog.init(project_id=project_id, base_url=BACKEND, experiment='fmnist_cnn_sweep')
print('project:', project_id)

CUDA available: True | NVIDIA GeForce RTX 3050 6GB Laptop GPU


project: proj_fashionmnist_cnn_sweep_3cddf661


### Dataset (real, versioned input)

In [2]:
DATASET_INFO

{'name': 'FashionMNIST',
 'version': 'v1',
 'preprocessing': 'ToTensor + normalize(mean=0.2860, std=0.3530)',
 'split_rationale': 'subset: 15000 train / 3000 val (fixed seed) for fast 10-run sweeps',
 'quality_issues': 'shirt/coat/pullover classes are visually confusable (inherent)'}

### The sweep — 10 configs across a wide hyperparameter space
Spans learning rate (5 orders of magnitude of effect), optimizer, capacity, dropout, and weight decay — so parameter-sensitivity has a clear signal.

In [3]:
BASE = dict(batch_size=128, weight_decay=0.0, dropout=0.0, conv_channels=32, hidden_dim=128, activation='relu', epochs=2, seed=0)
SWEEP = [
    {**BASE, 'lr':1e-3, 'optimizer':'adam'},
    {**BASE, 'lr':3e-3, 'optimizer':'adam'},
    {**BASE, 'lr':3e-4, 'optimizer':'adam'},
    {**BASE, 'lr':1e-2, 'optimizer':'adam'},
    {**BASE, 'lr':3e-2, 'optimizer':'adam'},
    {**BASE, 'lr':1e-3, 'optimizer':'adam', 'dropout':0.25},
    {**BASE, 'lr':1e-3, 'optimizer':'adam', 'conv_channels':48, 'hidden_dim':192},
    {**BASE, 'lr':1e-3, 'optimizer':'adam', 'weight_decay':5e-4},
    {**BASE, 'lr':5e-3, 'optimizer':'sgd'},
    {**BASE, 'lr':1e-1, 'optimizer':'sgd'},
]
len(SWEEP)

10

### Run the sweep
Each run: Pre-flight Guard → train on GPU → record the full picture (config, metrics, dataset, checkpoint artifact, **GPU-hours**).

In [4]:
EXPERIMENT = 'fmnist_cnn_sweep'
total_gpu_hours = 0.0
for i, cfg in enumerate(SWEEP, 1):
    if groundhog.check(cfg, experiment=EXPERIMENT).get('already_tried'):
        print(f'[{i:2d}/10] already tried lr={cfg["lr"]} {cfg["optimizer"]} — skipping'); continue
    out = os.path.join('outputs', EXPERIMENT, f'run_{i:02d}')
    m = train_and_evaluate(cfg, out_dir=out)
    total_gpu_hours += m['gpu_hours']
    groundhog.remember(config=cfg, metrics=m, experiment=EXPERIMENT, dataset=DATASET_INFO,
                       output_dir=out, gpu_hours=m['gpu_hours'], wall_clock_seconds=m['wall_clock_seconds'],
                       hypothesis=f"cfg #{i}: lr={cfg['lr']} opt={cfg['optimizer']}",
                       rationale=f"sweep run {i}: val_acc={m['val_accuracy']} in {m['wall_clock_seconds']}s on {m['device']}")
    print(f"[{i:2d}/10] lr={cfg['lr']:<6} {cfg['optimizer']:<4} -> val_acc={m['val_accuracy']:.4f} "
          f"loss={m['val_loss']:.3f} | {m['wall_clock_seconds']:.1f}s | {m['gpu_hours']*3600:.1f} gpu-sec")
print(f'\nTotal GPU-hours across sweep: {total_gpu_hours:.5f} h ({total_gpu_hours*3600:.0f} GPU-seconds)')

[ 1/10] lr=0.001  adam -> val_acc=0.8347 loss=0.442 | 6.3s | 6.3 gpu-sec


[ 2/10] lr=0.003  adam -> val_acc=0.8597 loss=0.372 | 6.4s | 6.4 gpu-sec


[ 3/10] lr=0.0003 adam -> val_acc=0.8137 loss=0.494 | 6.9s | 6.9 gpu-sec


[ 4/10] lr=0.01   adam -> val_acc=0.8213 loss=0.486 | 7.1s | 7.1 gpu-sec


[ 5/10] lr=0.03   adam -> val_acc=0.6727 loss=0.813 | 7.2s | 7.2 gpu-sec


[ 6/10] lr=0.001  adam -> val_acc=0.8453 loss=0.409 | 6.8s | 6.8 gpu-sec


[ 7/10] lr=0.001  adam -> val_acc=0.8630 loss=0.369 | 7.6s | 7.6 gpu-sec


[ 8/10] lr=0.001  adam -> val_acc=0.8320 loss=0.449 | 7.0s | 7.0 gpu-sec


[ 9/10] lr=0.005  sgd  -> val_acc=0.7947 loss=0.538 | 6.5s | 6.5 gpu-sec


[10/10] lr=0.1    sgd  -> val_acc=0.8630 loss=0.364 | 7.6s | 7.6 gpu-sec

Total GPU-hours across sweep: 0.01928 h (69 GPU-seconds)


### Ask the memory

In [5]:
print(groundhog.query('Which learning rate and optimizer gave the best val_accuracy on FashionMNIST, and which hyperparameter mattered most?'))

The best validation accuracy on FashionMNIST was achieved with a learning rate of **0.003** using the **Adam** optimizer, yielding a validation accuracy of **0.8637**. The most impactful hyperparameter was the learning rate (lr), which significantly influenced the validation accuracy by approximately **0.0863**.


### Derived insights — clear signal from 10 runs

In [6]:
ins = json.load(urllib.request.urlopen(f'{BACKEND}/api/insights?project_id={project_id}', timeout=60))
print('n_runs:', ins['n_runs'], '| completed:', ins['n_completed'])
print('summary:', ins['summary'])
print('\nparameter sensitivity (most -> least impactful):')
for s in ins['parameter_sensitivity']:
    print(f"  {s['parameter']:<14} impact={s['sensitivity']:<8} best={s['best_value']}  ({s['metric']})")

n_runs: 10 | completed: 10
summary: Across 10 completed runs: The most impactful hyperparameter is 'lr' (moves val_accuracy by ~0.1903); best value seen: 0.1. 'optimizer' had the least effect (0.0111). Best on FashionMNIST: val_accuracy=0.863.

parameter sensitivity (most -> least impactful):
  lr             impact=0.1903   best=0.1  (val_accuracy)
  hidden_dim     impact=0.0478   best=192  (val_accuracy)
  conv_channels  impact=0.0478   best=48  (val_accuracy)
  dropout        impact=0.0281   best=0.25  (val_accuracy)
  weight_decay   impact=0.0133   best=0.0005  (val_accuracy)
  optimizer      impact=0.0111   best=sgd  (val_accuracy)


### Cost summary (GPU-hours tracked per run)

In [7]:
runs = json.load(urllib.request.urlopen(f'{BACKEND}/api/runs/?project_id={project_id}', timeout=30))['runs']
tot = sum((r.get('gpu_hours') or 0) for r in runs)
print(f'{len(runs)} runs, total {tot:.5f} GPU-hours ({tot*3600:.0f} GPU-seconds)')
for r in sorted(runs, key=lambda r: -(r.get('metrics',{}).get('val_accuracy') or 0))[:3]:
    print(f"  best: val_acc={r['metrics'].get('val_accuracy')} lr={r['config'].get('lr')} {r['config'].get('optimizer')} gpu_h={r.get('gpu_hours')}")

10 runs, total 0.01928 GPU-hours (69 GPU-seconds)
  best: val_acc=0.863 lr=0.1 sgd gpu_h=0.00211
  best: val_acc=0.863 lr=0.001 adam gpu_h=0.0021
  best: val_acc=0.8597 lr=0.003 adam gpu_h=0.00178


Open **http://localhost:5173**, select **FashionMNIST CNN Sweep**, and see the runs, the metric-over-runs chart, parameter sensitivity, and GPU-hour cost.